In [1]:
import pandas as pd
import numpy as np
# Load datasets
moffitt = pd.read_csv('/kaggle/input/datasets/mohamedaln/moffitt-and-bambah-mukku-et-al-merfish-all-cells2/Moffitt_and_Bambah-Mukku_et_al_merfish_all_cells2.csv')
boundaries = pd.read_csv('/kaggle/input/cellboundaries-example-animal/cellboundaries_example_animal.csv')

# Check the key columns first
print("Moffitt Cell_ID samples:", moffitt['Cell_ID'].head(10).tolist())
print("Boundaries feature_uID samples:", boundaries['feature_uID'].head(10).tolist())

Moffitt Cell_ID samples: ['6749ccb4-2ed1-4029-968f-820a287f43c8', '6cac74bd-4ea7-4701-8701-42563cc65eb8', '9f29bd57-16a5-4b26-b9f5-37598809da9e', 'd7eb4e0b-276e-47e3-a55c-0b033180a2fe', '54434f3a-eba9-4aec-af35-c9d317ffa1d5', '6dcfb0ba-763f-4330-8ff6-aa55fd5d6ba6', '45fbf54d-35d2-4ac3-a42a-c59751e9d4f7', 'cb0b8dcf-a941-4766-bbc9-9e0271392c43', 'b288e78c-3693-4aad-b40f-a21ad62be583', '7fb27368-5a7a-4eeb-9f67-a1259d9144dd']
Boundaries feature_uID samples: ['1f9a8c19-b089-43d1-b609-7e791dc2c70f', 'b13e98f4-5c2b-4e96-985e-3e93aedc7221', 'd06cb29e-10ee-4bbc-b74e-90237999ef4b', '12e2a165-57c7-4f37-96dd-23f6574af4ba', '14a0f396-b13d-4d45-89a3-86c2047bf3f9', '916a9888-e25e-4e58-9c11-7d44c8392b96', 'ea2d1046-4838-4916-a5fd-0c0274460c2b', 'b6382de6-9656-49a2-89c6-8becf1ecdbba', 'cfe60a3a-92a7-46d6-a2cf-687fd2af96fc', '4da51944-deb2-433c-843a-dfeb953186a3']


In [2]:
moffitt_ids = set(moffitt['Cell_ID'])
boundary_ids = set(boundaries['feature_uID'])
overlap = moffitt_ids & boundary_ids

print(f"Moffitt unique cells: {len(moffitt_ids)}")
print(f"Boundary unique cells: {len(boundary_ids)}")
print(f"Overlapping cells: {len(overlap)}")

Moffitt unique cells: 1027848
Boundary unique cells: 73655
Overlapping cells: 73655


In [3]:
# Check boundary lengths
matched = (boundaries['boundaryX'].str.count(';') == boundaries['boundaryY'].str.count(';')).sum()
print(f"Matched: {matched}")
print(f"Mismatched: {len(boundaries) - matched}")

Matched: 36852
Mismatched: 36803


In [4]:
# Keep only valid boundaries
boundaries = boundaries[boundaries['boundaryX'].str.count(';') == boundaries['boundaryY'].str.count(';')]
print(f"Valid boundaries: {len(boundaries)}")

Valid boundaries: 36852


In [5]:
# Check overlap between moffitt and valid boundaries
overlap = set(moffitt['Cell_ID']) & set(boundaries['feature_uID'])
print(f"Moffitt cells: {moffitt['Cell_ID'].nunique()}")
print(f"Valid boundary cells: {boundaries['feature_uID'].nunique()}")
print(f"Overlapping cells: {len(overlap)}")

Moffitt cells: 1027848
Valid boundary cells: 36852
Overlapping cells: 36852


In [6]:
# =============================================================
# Goal: Rebuild the merged_moffitt_boundaries_cleaned dataset
# using ONLY cells with equal X/Y boundary lengths.
# This ensures all boundary polygons are biologically valid.
# =============================================================

import pandas as pd
import numpy as np

# Step 1: Load both datasets
boundaries = pd.read_csv("/kaggle/input/cellboundaries-example-animal/cellboundaries_example_animal.csv")
moffitt = pd.read_csv("/kaggle/input/datasets/mohamedaln/moffitt-and-bambah-mukku-et-al-merfish-all-cells2/Moffitt_and_Bambah-Mukku_et_al_merfish_all_cells2.csv")

# Step 2: Keep only boundaries with matching X/Y lengths
boundaries_clean = boundaries[boundaries['boundaryX'].str.count(';') == boundaries['boundaryY'].str.count(';')]
print(f"Valid boundaries: {len(boundaries_clean)}")

# Step 3: Merge with Moffitt
merged = moffitt.merge(boundaries_clean, left_on='Cell_ID', right_on='feature_uID', how='inner')
print(f"After merge: {merged.shape}")

# Step 5: Remove Blank columns
blank_cols = [c for c in merged.columns if c.startswith('Blank_')]
merged = merged.drop(columns=blank_cols)
print(f"After removing Blank cols: {merged.shape}")

# Step 6: Remove Ambiguous cell class
merged = merged[merged['Cell_class'] != 'Ambiguous']
print(f"After removing Ambiguous: {merged.shape}")

# Step 7: Reorder columns — meta + boundary + genes
meta_cols = ['Cell_ID', 'Animal_ID', 'Animal_sex', 'Behavior', 'Bregma', 'Centroid_X', 'Centroid_Y', 'Cell_class', 'Neuron_cluster_ID']
boundary_cols = ['feature_uID', 'boundaryX', 'boundaryY']
gene_cols = [c for c in merged.columns if c not in meta_cols + boundary_cols]
merged = merged[meta_cols + boundary_cols + gene_cols]

# Step 8: Final summary
print(f"\n=== Final Dataset ===")
print(f"Shape: {merged.shape}")
print(f"Unique cells: {merged['Cell_ID'].nunique()}")
print(f"Cell classes ({merged['Cell_class'].nunique()}):")
print(merged['Cell_class'].value_counts())
print(f"\nBregma values: {sorted(merged['Bregma'].unique())}")
print(f"NaNs (excl Neuron_cluster_ID): {merged.drop(columns=['Neuron_cluster_ID']).isna().any().any()}")

# Step 9: Save
merged.to_csv('merged_moffitt_boundaries_cleaned_v2.csv', index=False)
print("\nSaved: merged_moffitt_boundaries_cleaned_v2.csv")

Valid boundaries: 36852
After merge: (36852, 173)
After removing Blank cols: (36852, 168)
After removing Ambiguous: (32648, 168)

=== Final Dataset ===
Shape: (32648, 168)
Unique cells: 32648
Cell classes (15):
Cell_class
Inhibitory       11330
Excitatory        5383
Astrocyte         4822
OD Mature 2       3229
Endothelial 1     2377
OD Immature 1     1298
Ependymal         1040
Microglia          886
Endothelial 3      800
OD Mature 1        535
Pericytes          344
Endothelial 2      332
OD Mature 4        180
OD Immature 2       59
OD Mature 3         33
Name: count, dtype: int64

Bregma values: [np.float64(-0.29), np.float64(-0.24), np.float64(-0.19), np.float64(-0.14), np.float64(-0.09), np.float64(-0.04), np.float64(0.01), np.float64(0.06), np.float64(0.11), np.float64(0.16), np.float64(0.21), np.float64(0.26)]
NaNs (excl Neuron_cluster_ID): True

Saved: merged_moffitt_boundaries_cleaned_v2.csv


In [7]:
# Find which columns have NaNs (excluding Neuron_cluster_ID)
nan_cols = merged.drop(columns=['Neuron_cluster_ID']).columns[merged.drop(columns=['Neuron_cluster_ID']).isna().any()]
print(f"Columns with NaNs: {nan_cols.tolist()}")
for c in nan_cols:
    print(f"  {c}: {merged[c].isna().sum()} NaNs")

Columns with NaNs: ['Fos']
  Fos: 32648 NaNs
